# XGBoost multiclass (5 gait anomaly classes)

This notebook trains an XGBoost **multiclass** model that predicts one of the 5 high-level gait anomaly classes
defined via `CLASS_MAP`.

**Notes**
- The preprocessing + feature extraction pipeline is reused from `Pose_Preprocessing_Pipeline_2` and `feature_extraction_cleaned`.
- Windows with *no* fine-grained labels are treated as `normal` and are **dropped by default** (since this is a 5-class *anomaly-type* model).
- You can flip `INCLUDE_NORMAL = True` to train a 6-class model including `normal`.


In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix


import sys
from pathlib import Path
import xgboost as xgb

# repo root = parent of the notebooks folder
repo_root = Path.cwd().resolve().parent
sys.path.insert(0, str(repo_root))

from modeling.feature_extraction_cleaned import extract_features_from_windows, FeatureConfig
from modeling.Pose_Preprocessing_Pipeline_2 import add_pose_column, preprocess_gait_sliding_windows, apply_qc_windows, GAIT_JOINTS

pd.set_option("display.max_columns", 200)


##  Define the 5 gait anomaly classes

We map fine-grained labels (from the pipeline) into 5 biomechanically motivated high-level classes.

In [2]:
# ---------------------------------------------------------------------
# High-level 5-class gait anomaly label scheme
# ---------------------------------------------------------------------
CLASS_MAP = {
    "gait_anomaly_distal_foot_control_deficit": {
        "Foot Drop",
        "Foot Slap",
        "Inadequate Dorsiflexion",
        "Foot Flat Initial Contact",
        "Excess Pronation",
        "Excess Supination",
        "Reduced Metatarsophalangeal Joint Extension",
        "Absent Heel Rise During Terminal Stance",
        "Early Heel Rise",
        "Steppage Gait",
    },
    "gait_anomaly_knee_sagittal_plane_abnormality": {
        "Knee Extensor Thrust",
        "Knee Hyperextension",
        "Reduced Knee Extension",
        "Reduced Knee Flexion",
        "Knee Valgus",
    },
    "gait_anomaly_hip_pelvic_control_deficit": {
        "Trendelenburg",
        "Hip Hiking",
        "Posterior Pelvic Tilt",
        "Anterior Pelvic Tilt",
        "Reduced Pelvic Rotation",
        "Reduced Hip Extension",
        "Reduced Hip Internal Rotation",
        "Circumduction",
        "Medial Whip",
    },
    "gait_anomaly_trunk_balance_abnormality": {
        "Reduced Arm Swing",
        "Forward Lean",
        "Left Lean",
        "Right Lean",
        "Reduced Trunk Rotation",
        "Imbalance",
        "Cautious Gait",
    },
    "gait_anomaly_spatiotemporal_asymmetry": {
        "Wide Base of Support",
        "Step Length Asymmetry",
        "Reduced Step Length",
        "Reduced Left Weightshift",
    },
}

# Reverse lookup: fine label -> coarse class
FINE_TO_COARSE = {
    fine_label: coarse_label
    for coarse_label, fine_labels in CLASS_MAP.items()
    for fine_label in fine_labels
}

COARSE_CLASSES_5 = list(CLASS_MAP.keys())
COARSE_TO_ID_5 = {c: i for i, c in enumerate(COARSE_CLASSES_5)}
ID_TO_COARSE_5 = {i: c for c, i in COARSE_TO_ID_5.items()}

INCLUDE_NORMAL = False  # set True to train 6 classes: 5 anomalies + normal
NORMAL_LABEL = "normal"


## Load data (Polars)

In [74]:
df_raw = pl.read_parquet("../data/clean_gait_data.parquet")
print("df_raw:", df_raw.shape)


df_raw: (9251704, 32)


## Build video-level structure + patient mapping
`add_pose_column` gibt (in deiner Version) **nur** `df_video_pl` zurück.

In [75]:
df_video_pl = add_pose_column(df_raw)
print("df_video_pl:", df_video_pl.shape)
print("Columns:", df_video_pl.columns)

# Patient mapping MUST be based on the same key used in window_ids.
# In your data, window_ids look like: 'semantic_segmentation_..._DensePose_landmarks_win000_f0-59'
# and df_video_pl contains a matching 'video_id' string column.
video_to_patient = (
    df_video_pl
    .select(["video_id", "patient_name"])
    .unique()
    .to_pandas()
    .set_index("video_id")["patient_name"]
    .to_dict()
)

print("n videos in mapping:", len(video_to_patient))


df_video_pl: (3279, 33)
Columns: ['id', 'patient_name', 'frame', 'movement_type', 'side', 'timestamp_ms', 'landmark_id', 'x_norm', 'y_norm', 'z_norm', 'visibility', 'x_px', 'y_px', 'dataset', 'gait_pattern', 'add_pattern_info', 'title', 'fps', 'width', 'height', 'gait_markers', 'file_path', 'video_id', 'dataset_encoded', 'gait_anomalies', 'row_nr', 'null', 'gait_anomaly_knee_sagittal_plane_abnormality', 'gait_anomaly_trunk_balance_abnormality', 'gait_anomaly_spatiotemporal_asymmetry', 'gait_anomaly_hip_pelvic_control_deficit', 'gait_anomaly_distal_foot_control_deficit', 'pose']
n videos in mapping: 3279


## Sliding windows, QC, and feature extraction

In [76]:
# window params (match your pipeline)
window_seconds = 2.0
overlap = 0.5
resample_frames = 60

X_windows, y_binary, y_multilabel, window_ids = preprocess_gait_sliding_windows(
    df_video_pl,
    window_seconds=window_seconds,
    overlap=overlap,
    resample_frames=resample_frames,
)

fps_effective = resample_frames / window_seconds  # e.g. 30 Hz
X_clean, y_binary_clean, y_multilabel_clean, window_ids_clean, qc_df = apply_qc_windows(
    X_windows, y_binary, y_multilabel, window_ids, fps=fps_effective
)

print("Windows before QC:", X_windows.shape[0])
print("Windows after QC :", X_clean.shape[0])
print("y_multilabel_clean shape:", y_multilabel_clean.shape)
print("Positive windows:", int((y_multilabel_clean.sum(axis=1) > 0).sum()))
print("Positives per label index:", y_multilabel_clean.sum(axis=0))
print("Example window_ids:", window_ids_clean[:3])


QC-clean windows: 16048 / 16829 (95.36%)
Windows before QC: 16829
Windows after QC : 16048
y_multilabel_clean shape: (16048, 5)
Positive windows: 893
Positives per label index: [323 357 361 346 258]
Example window_ids: ['semantic_segmentation_PA205_UGS_WJ_2_DensePose_landmarks_win000_f0-59'
 'semantic_segmentation_PA205_UGS_WJ_2_DensePose_landmarks_win001_f33-92'
 'semantic_segmentation_PA205_UGS_WJ_2_DensePose_landmarks_win002_f66-125']


In [77]:
# Map window_id -> patient_name via video_id prefix (before '_win...')
def video_key_from_window_id(wid: str) -> str:
    return wid.split("_win")[0]

window_patient_name = [video_to_patient.get(video_key_from_window_id(w)) for w in window_ids_clean]

import pandas as pd
print("patient_name null count:", pd.isna(window_patient_name).sum(), "/", len(window_patient_name))
print("example patients:", window_patient_name[:5])


patient_name null count: 0 / 16048
example patients: ['PA205', 'PA205', 'PA205', 'PA205', 'PA205']


In [78]:
# Convert reduced joints back to full 33 for feature extractor
N, T, Jg, C = X_clean.shape
X_full = np.full((N, T, 33, 3), np.nan, dtype=np.float32)
X_full[:, :, GAIT_JOINTS, :] = X_clean

cfg = FeatureConfig()
df_features = extract_features_from_windows(
    X_windows=X_full,
    fps=fps_effective,
    gait_pattern=None,
    movement_type=None,
    side=None,
    source_file=None,
    cfg=cfg,
)
print("df_features:", df_features.shape)
df_features.head()


df_features: (16048, 88)


,step_height_L,step_height_R,step_length_L,step_length_R,pelvis_drop_mean,pelvis_drop_std,trunk_lean_mean,trunk_lean_std,heel_range_L,heel_range_R,step_height_symmetry,step_length_symmetry,knee_L_moving_time_sec,knee_L_still_time_sec,knee_L_moving_fraction,knee_L_still_fraction,knee_L_mean_speed,knee_L_max_speed,knee_L_total_time_sec,knee_R_moving_time_sec,knee_R_still_time_sec,knee_R_moving_fraction,knee_R_still_fraction,knee_R_mean_speed,knee_R_max_speed,knee_R_total_time_sec,knee_L_rom_y,knee_R_rom_y,hip_L_rom_y,hip_R_rom_y,shoulder_L_rom_x,shoulder_R_rom_x,ankle_L_rom_y,ankle_R_rom_y,knee_rom_asym,hip_rom_asym,shoulder_rom_asym,ankle_rom_asym,ankle_L_moving_fraction,ankle_L_still_fraction,ankle_R_moving_fraction,ankle_R_still_fraction,stance_ratio_L,stance_ratio_R,stance_ratio_asym,knee_angle_L_mean,knee_angle_L_std,knee_angle_L_rom,knee_angle_R_mean,knee_angle_R_std,knee_angle_R_rom,hip_angle_L_mean,hip_angle_L_std,hip_angle_L_rom,hip_angle_R_mean,hip_angle_R_std,hip_angle_R_rom,ankle_angle_L_mean,ankle_angle_L_std,ankle_angle_L_rom,ankle_angle_R_mean,ankle_angle_R_std,ankle_angle_R_rom,knee_angle_rom_asym,hip_angle_rom_asym,ankle_angle_rom_asym,step_L_mean_step_time,step_L_std_step_time,step_L_cadence,step_L_mean_stride_time,step_L_std_stride_time,step_L_step_time_cv,step_R_mean_step_time,step_R_std_step_time,step_R_cadence,step_R_mean_stride_time,step_R_std_stride_time,step_R_step_time_cv,step_time_asym,cadence_asym,step_width_mean,step_width_std,label_fine,label_class,label_id,movement_type,side,source_file
0,0.398172,0.415597,0.759398,0.922266,-0.010791,0.011384,0.014134,0.064486,0.525217,0.600357,-0.021413,-0.096850,1.966667,0.0,1.0,0.0,1.604280,4.294055,1.966667,1.966667,0.0,1.0,0.0,1.383606,4.308143,1.966667,0.181119,0.144105,0.024925,0.024925,0.146707,0.182955,0.398172,0.415597,0.113810,0.0,-0.109956,-0.021413,1.0,0.0,1.0,0.0,0.0,0.0,0.0,159.067307,14.073471,60.898361,162.309952,12.132642,53.870239,158.534607,10.825083,38.965393,165.820724,5.753232,29.011673,108.032967,12.766478,57.577240,95.388885,12.251139,53.873993,0.061237,0.146428,0.033228,0.441667,0.138193,135.849057,0.855556,0.181217,0.312889,0.458333,0.165622,130.909091,0.955556,0.159474,0.361356,-0.018518,0.018519,0.328856,0.237881,None,None,None,None,None,None
1,0.413256,0.437916,0.723399,0.906658,-0.013177,0.009357,0.020031,0.041325,0.550870,0.640145,-0.028972,-0.112425,1.966667,0.0,1.0,0.0,1.338574,3.631544,1.966667,1.966667,0.0,1.0,0.0,1.408793,3.308523,1.966667,0.133495,0.152424,0.024696,0.024696,0.115134,0.132378,0.413256,0.437916,-0.066205,0.0,-0.069671,-0.028972,1.0,0.0,1.0,0.0,0.0,0.0,0.0,161.343964,14.414484,62.074074,159.643982,16.295225,68.364235,158.507736,8.949921,28.921402,165.258896,7.468611,42.286346,103.065651,10.711372,42.764641,92.897087,12.709955,43.823112,-0.048223,-0.187689,-0.012224,0.340000,0.024944,176.470588,0.675000,0.014434,0.073366,0.441667,0.103749,135.849057,0.911111,0.056656,0.234904,-0.130064,0.130064,0.321874,0.239016,None,None,None,None,None,None
2,0.308254,0.379043,0.753881,0.759665,-0.011071,0.006292,-0.011824,0.038741,0.465184,0.565969,-0.102995,-0.003822,1.966667,0.0,1.0,0.0,1.257667,3.460015,1.966667,1.966667,0.0,1.0,0.0,1.302001,3.336690,1.966667,0.181727,0.146564,0.016260,0.016260,0.077875,0.092454,0.308254,0.379043,0.107109,0.0,-0.085591,-0.102995,1.0,0.0,1.0,0.0,0.0,0.0,0.0,161.584473,11.902147,44.115295,161.782898,15.554506,65.940460,157.862152,8.230358,27.870804,168.421112,8.140394,41.919296,103.024734,14.131269,55.818886,94.518944,12.372847,53.651558,-0.198310,-0.201296,0.019798,0.425000,0.101036,141.176471,0.844444,0.068493,0.237732,0.425000,0.086201,141.176471,0.866667,0.098131,0.202825,0.000000,0.000000,0.291804,0.219606,None,None,None,None,None,None
3,0.278341,0.392868,0.748501,0.825379,-0.009896,0.006478,-0.048086,0.027736,0.421678,0.595906,-0.170627,-0.048846,1.966667,0.0,1.0,0.0,1.239449,4.004086,1.966667,1.966667,0.0,1.0,0.0,1.225369,3.677227,1.966667,0.161622,0.146250,0.016678,0.016678,0.095009,0.09

##  Define 5 classes (index order!)
 `y_multilabel_clean` ist ein Multihot-Vektor der Länge 5. **Die Reihenfolge der 5 Indizes muss stimmen.**

Bei dir existieren Spalten in `df_raw` wie `gait_anomaly_knee_sagittal_plane_abnormality`, etc.
Falls die Reihenfolge unten nicht passt, ändere einfach die Liste `INDEX_TO_CLASS`.


In [79]:
# IMPORTANT: Ensure this order matches the multi-hot vector order in y_multilabel_clean.
# If you are unsure, compare with df_raw label columns or your pipeline definition.
INDEX_TO_CLASS = [
    "gait_anomaly_knee_sagittal_plane_abnormality",
    "gait_anomaly_trunk_balance_abnormality",
    "gait_anomaly_spatiotemporal_asymmetry",
    "gait_anomaly_hip_pelvic_control_deficit",
    "gait_anomaly_distal_foot_control_deficit",
]

NORMAL_LABEL = "normal"
CLASS_TO_ID = {c: i for i, c in enumerate(INDEX_TO_CLASS)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}

print("Classes:")
for i, c in enumerate(INDEX_TO_CLASS):
    print(i, c)


Classes:
0 gait_anomaly_knee_sagittal_plane_abnormality
1 gait_anomaly_trunk_balance_abnormality
2 gait_anomaly_spatiotemporal_asymmetry
3 gait_anomaly_hip_pelvic_control_deficit
4 gait_anomaly_distal_foot_control_deficit


## Multihot → Single-label (Multiclass)
Policy:
- Keine 1 im Multihot → `normal`
- Eine oder mehrere 1 → wähle per Priorität (Index-Reihenfolge).


In [80]:
def multihot_to_single_label(row: np.ndarray) -> str:
    idx = np.flatnonzero(row)
    if len(idx) == 0:
        return NORMAL_LABEL
    # tie-break by priority order = first index
    return INDEX_TO_CLASS[int(idx[0])]

coarse_labels = [multihot_to_single_label(r) for r in y_multilabel_clean]

df_mc = df_features.copy()
df_mc["label_coarse"] = coarse_labels
df_mc["patient_name"] = window_patient_name

print("label_coarse counts:")
print(df_mc["label_coarse"].value_counts(dropna=False))


label_coarse counts:
label_coarse
normal                                          15155
gait_anomaly_knee_sagittal_plane_abnormality      323
gait_anomaly_spatiotemporal_asymmetry             230
gait_anomaly_trunk_balance_abnormality            220
gait_anomaly_hip_pelvic_control_deficit           120
Name: count, dtype: int64


##  Build training set (drop normal) + X/y/groups
Wir trainieren das **5-class anomaly-type** Modell, daher droppen wir `normal`.


In [81]:
# keep only anomaly windows + valid patient
df_train = df_mc[(df_mc["label_coarse"] != NORMAL_LABEL) & (df_mc["patient_name"].notna())].copy()
print("df_train:", df_train.shape)
print("unique patients:", df_train["patient_name"].nunique())
print("label counts:", df_train["label_coarse"].value_counts().to_dict())

# label_id
df_train["label_id"] = df_train["label_coarse"].map(CLASS_TO_ID)
before = len(df_train)
df_train = df_train[df_train["label_id"].notna()].copy()
df_train["label_id"] = df_train["label_id"].astype(int)
print("dropped unmapped:", before - len(df_train))

# feature cols (numeric only)
non_feature_cols = {"patient_name", "label_coarse", "label_id"}
feature_cols = [c for c in df_train.columns if c not in non_feature_cols and pd.api.types.is_numeric_dtype(df_train[c])]
print("n feature cols:", len(feature_cols))

X = df_train[feature_cols].copy()
X = X.fillna(X.median(numeric_only=True))

y = df_train["label_id"].to_numpy()
groups = df_train["patient_name"].to_numpy()

print("len(X):", len(X), "len(y):", len(y), "len(groups):", len(groups))


df_train: (893, 90)
unique patients: 98
label counts: {'gait_anomaly_knee_sagittal_plane_abnormality': 323, 'gait_anomaly_spatiotemporal_asymmetry': 230, 'gait_anomaly_trunk_balance_abnormality': 220, 'gait_anomaly_hip_pelvic_control_deficit': 120}
dropped unmapped: 0
n feature cols: 82
len(X): 893 len(y): 893 len(groups): 893


## Patient-wise split (GroupShuffleSplit) + Guards

In [82]:
import pandas as pd
assert len(X) == len(y) == len(groups), "X/y/groups length mismatch"
assert pd.isna(groups).sum() == 0, "groups contains NaNs"
assert pd.Series(groups).nunique() >= 2, "Need at least 2 patients"

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train patients:", pd.Series(groups[train_idx]).nunique())
print("Test patients :", pd.Series(groups[test_idx]).nunique())
print("Overlap:", set(groups[train_idx]) & set(groups[test_idx]))

print("Train label counts:", pd.Series(y_train).value_counts().sort_index().to_dict())
print("Test label counts :", pd.Series(y_test).value_counts().sort_index().to_dict())


Train: (693, 82) Test: (200, 82)
Train patients: 78
Test patients : 20
Overlap: set()
Train label counts: {0: 211, 1: 197, 2: 182, 3: 103}
Test label counts : {0: 112, 1: 23, 2: 48, 3: 17}


## Train XGBoost muliclass + evaluation

In [85]:

labels = list(range(num_class))  # z.B. [0,1,2,3,4]
target_names = [ID_TO_CLASS[i] for i in labels]

print(
    classification_report(
        y_test,
        y_pred,
        labels=labels,
        target_names=target_names,
        zero_division=0,   # wichtig: verhindert Crash bei fehlenden Klassen
    )
)

print("Confusion matrix:\n", confusion_matrix(y_test, y_pred, labels=labels))


                                              precision    recall  f1-score   support

gait_anomaly_knee_sagittal_plane_abnormality       0.88      0.76      0.81       112
      gait_anomaly_trunk_balance_abnormality       0.68      0.57      0.62        23
       gait_anomaly_spatiotemporal_asymmetry       0.64      0.94      0.76        48
     gait_anomaly_hip_pelvic_control_deficit       0.93      0.76      0.84        17
    gait_anomaly_distal_foot_control_deficit       0.00      0.00      0.00         0

                                   micro avg       0.78      0.78      0.78       200
                                   macro avg       0.63      0.61      0.61       200
                                weighted avg       0.80      0.78      0.78       200

Confusion matrix:
 [[85  5 21  1  0]
 [ 6 13  4  0  0]
 [ 3  0 45  0  0]
 [ 3  1  0 13  0]
 [ 0  0  0  0  0]]


In [87]:
print("y_test unique:", np.unique(y_test))
print("y_pred unique:", np.unique(y_pred))


y_test unique: [0 1 2 3]
y_pred unique: [0 1 2 3]


## Save model + metadata 

In [86]:
from pathlib import Path
import json

out_dir = Path("models")
out_dir.mkdir(exist_ok=True)

model_path = out_dir / "xgboost_gait_5class.bin"
model.save_model(model_path)

meta = {
    "classes": INDEX_TO_CLASS,
    "class_to_id": CLASS_TO_ID,
    "id_to_class": ID_TO_CLASS,
    "feature_cols": feature_cols,
    "normal_label": NORMAL_LABEL,
    "label_policy": "multihot->single via first positive index (priority order)",
}
meta_path = out_dir / "xgboost_gait_5class_metadata.json"
meta_path.write_text(json.dumps(meta, indent=2))

print("Saved model to:", model_path.resolve())
print("Saved metadata to:", meta_path.resolve())


Saved model to: C:\Users\lejaz\spice_bootcamp\GAITy-Capstone-Modeling\notebooks\models\xgboost_gait_5class.bin
Saved metadata to: C:\Users\lejaz\spice_bootcamp\GAITy-Capstone-Modeling\notebooks\models\xgboost_gait_5class_metadata.json


c:\Users\lejaz\spice_bootcamp\GAITy-Capstone-Modeling\.venv\Lib\site-packages\xgboost\sklearn.py:1118: UserWarning: [18:31:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
